## Neural Network (NN) - `MLPClassifier`

All training code called here.

- Training Time Taken
- Test Accuracy

### Model Parameters

For the Neural Network, the `MLPClassifier` will use one single hidden layer with $20, 100$ nodes, and learning rates of $0.001, 0.01, 0.1$ will be evaluated using cross-validation.

### Load Shared Library

`autoreload` enables it to reload all changes in `shared_library.py`


In [1]:
%load_ext autoreload
%autoreload 2 

# import everything!!
from shared_library import * 

### Output Details

Results for neural network are captured in `/results/nn_results_{SOLVER}_{ACTIVATION}_{yyyymmddhhmm()}.csv`


### Experiments

Perform a baseline with a smaller set and compare, identify how long it takes to classify a 50-1000 images.

Load a set of 50, 100, 200, 500, 1000 and run a `MLPClassifier` to identify time taken. Do some sample processing to tune `max_iter`, `learning_rate_init` or any other parameters


In [ ]:
TIME_OF_EXECUTION = yyyymmddhhmm()
RESULTS_CSV = f"./results/nn_results_adam_relu_{yyyymmddhhmm()}.csv"
OUTPUT_LOG = f"./results/linear_experiments_results_{TIME_OF_EXECUTION}"
N_JOBS = 1

X, y = load_all_images_flat(TRAIN_DATA_PATH, TRAIN_LABEL_PATH)

print(X.shape, X.dtype, X.min(), X.max())
print(y.shape, y.dtype, y.min(), y.max())


def mlp(hidden_layer, regularization):
    print(f"Running for layers:{hidden_layer}, @: {regularization}")

    start_time = perf_counter()

    clf = MLPClassifier(
        solver="adam",
        activation="relu",
        hidden_layer_sizes=hidden_layer,    
        max_iter=10000,
        alpha=regularization,
        learning_rate_init=0.0001,
        verbose=True,
        early_stopping=True,
        random_state=SEED,
    )

    # This is only extra line that `cross_val_score` calls, just emulating it. Too lazy to check if skipping `check_scoring` does something extra.
    scoring = check_scoring(estimator=clf, scoring="balanced_accuracy")

    with open(f"{OUTPUT_LOG}_{hidden_layer}_{regularization}.log", "w") as logfile:
        with pipes(stdout=logfile, stderr=logfile):
            print(
                f"====== Running layers: {hidden_layer} @:{regularization} ====== ",
                flush=True,
            )
            scores = cross_validate(
                X=X,
                y=y,
                estimator=clf,
                cv=5,
                scoring=scoring,
                return_estimator=True,
                return_train_score=True,
                n_jobs=N_JOBS,
            )
    end_time = perf_counter()

    test_score = scores["test_score"]
    train_score = scores["train_score"]
    fit_time = scores["fit_time"]
    score_time = scores["score_time"]
    estimator = scores["estimator"]

    return {
        "hidden_layer": hidden_layer,
        "max_iters": 100000,
        "regularization": regularization,
        "test_score": test_score,
        "train_score": train_score,
        "fit_time": fit_time,
        "score_time": score_time,
        # "estimator": estimator,
        "execution_time": end_time - start_time,
    }


hidden_layers = [20, 100, (20, 100)]
regularization = [0.001, 0.01, 0.1]

experiments = list(product(hidden_layers, regularization))

# init results
rows = []
df = pd.DataFrame()

print(f"Running {len(experiments)} experiments")

Image Shape:  (5000, 27648) float32 0.0 1.0
(5000, 27648) float32 0.0 1.0
(5000,) uint8 1 10
Running 9 experiments


In [3]:
from IPython.display import clear_output

for la, r in experiments:
    row = mlp(la, r)
    rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)  # flush to disk immediately
    clear_output(wait=True)
    display(df)

print("This experiment was last executed on: ", datetime.datetime.now())

,hidden_layer,max_iters,regularization,test_score,train_score,fit_time,score_time,execution_time
0,20,100000,0.001,"[0.194, 0.24699999999999997, 0.257, 0.29, 0.27...","[0.257, 0.298, 0.3097499999999999, 0.346250000...","[3.1415250301361084, 2.947065830230713, 4.9481...","[0.03505730628967285, 0.01209115982055664, 0.0...",18.245733
1,20,100000,0.010,"[0.193, 0.248, 0.258, 0.275, 0.24]","[0.25575000000000003, 0.298, 0.324499999999999...","[2.987457036972046, 2.834861993789673, 5.51314...","[0.008028984069824219, 0.007725954055786133, 0...",17.590969
2,20,100000,0.100,"[0.193, 0.24300000000000002, 0.252, 0.28, 0.273]","[0.25549999999999995, 0.29625, 0.2799999999999...","[2.769443988800049, 2.7347240447998047, 3.3730...","[0.016015052795410156, 0.007821083068847656, 0...",17.297733
3,100,100000,0.001,"[0.356, 0.345, 0.332, 0.396, 0.37300000000000005]","[0.4585, 0.41500000000000004, 0.374, 0.6027500...","[7.929963827133179, 4.066725254058838, 3.37184...","[0.009902000427246094, 0.009741783142089844, 0...",32.355328
4,100,100000,0.010,"[0.358, 0.34299999999999997, 0.364000000000000...","[0.40125, 0.42824999999999996, 0.5892499999999...","[4.704674959182739, 4.785542011260986, 12.6274...","[0.009860992431640625, 0.010086774826049805, 0...",44.179310
5,100,100000,0.100,"[0.35700000000000004, 0.36, 0.3590000000000000...","[0.47025000000000006, 0.53975, 0.58275, 0.6039...","[8.960913181304932, 8.976558923721313, 12.6698...","[0.04536104202270508, 0.011739969253540039, 0....",55.513632
6,"(20, 100)",100000,0.001,"[0.30300000000000005, 0.286, 0.269, 0.27, 0.16]","[0.372, 0.33799999999999997, 0.281249999999999...","[8.357964038848877, 6.997035026550293, 4.81441...","[0.013033866882324219, 0.007234811782836914, 0...",32.850318
7,"(20, 100)",100000,0.010,"[0.301, 0.253, 0.175, 0.256, 0.16499999999999998]","[0.37725, 0.26825, 0.18000000000000002, 0.2917...","[11.365559816360474, 5.184144973754883, 3.6040...","[0.021504878997802734, 0.01983499526977539, 0....",31.959949
8,"(20, 100)",100000,0.100,"[0.28200000000000003, 0.259, 0.223, 0.27399999...","[0.30924999999999997, 0.308, 0.25475, 0.333000...","[5.422261953353882, 7.09723687171936, 7.473909...","[0.018932104110717773, 0.012231111526489258, 0...",36.066713


This experiment was last executed on:  2026-10-16 10:33:40.416181


### FINAL - Fit Test - running against 8K test data

Model with `hidden_layers` = $20, 100$ and `learning_rate` = $REPLACE$ has the maximum **Mean Accuracy** of $0.REPLACE$, and we'll use that to do a `fit` test against train data


In [5]:
TIME_OF_EXECUTION = yyyymmddhhmm()
RESULTS_CSV = f"./results/nn_results_adam_relu_{yyyymmddhhmm()}.csv"
OUTPUT_LOG = f"./results/linear_experiments_results_{TIME_OF_EXECUTION}"
N_JOBS = 1

# Load Training Data ~ 5K records
X_train, y_train = load_all_images_flat(TRAIN_DATA_PATH, TRAIN_LABEL_PATH)

print(X_train.shape, X_train.dtype, X_train.min(), X_train.max())
print(y_train.shape, y_train.dtype, y_train.min(), y_train.max())

# Load Test Data ~ 8K records
X_test, y_test = load_all_images_flat(TEST_DATA_PATH, TEST_LABEL_PATH)

print(X_test.shape, X_test.dtype, X_test.min(), X_test.max())
print(y_test.shape, y_test.dtype, y_test.min(), y_test.max())

hidden_layer = 100
regularization = 0.001

start_time = perf_counter()

clf = MLPClassifier(
    solver="adam",
    activation="relu",
    hidden_layer_sizes=hidden_layer,    
    max_iter=10000,
    alpha=regularization,
    learning_rate_init=0.0001,
    verbose=True,
    early_stopping=True,
    random_state=SEED,
).fit(X_train, y_train)

# Run an accuracy scoring.
accuracy_score = clf.score(X_test, y_test)

end_time = perf_counter()

output = {
    "hidden_layer": hidden_layer,
    "max_iters": 100000,
    "regularization": regularization,
    "accuracy_score": accuracy_score,
    "execution_time": end_time - start_time,
}

rows = []
df = pd.DataFrame()

rows.append(output)
df = pd.DataFrame(rows)
df.to_csv(RESULTS_CSV, index=False)  # flush to disk immediately
clear_output(wait=True)
display(df)

print("This experiment was last executed on: ", datetime.datetime.now())

,hidden_layer,max_iters,regularization,accuracy_score,execution_time
0,100,100000,0.001,0.36325,7.524407


This experiment was last executed on:  2026-10-16 10:35:37.515127
